# Paso 2: Obtención y Carga del Conjunto de Datos

El dataset utilizado es **Online Learning Engagement Dataset**, disponible públicamente en [Kaggle](https://www.kaggle.com/).

Se descarga mediante la **API pública de Kaggle**, que es una API REST oficial que permite acceder a datasets, competiciones y kernels de forma programática. Esto cumple con el requisito de obtener los datos a través de una API pública.

### Configuración previa de la API de Kaggle
1. Crear una cuenta en [kaggle.com](https://www.kaggle.com)
2. Ir a **Account → API → Create New API Token** → descarga `kaggle.json`
3. Colocar el archivo en `~/.kaggle/kaggle.json` (Mac/Linux) o `C:\Users\<user>\.kaggle\kaggle.json` (Windows)
4. Ejecutar: `pip install kaggle`

In [ ]:
## 2.1 Instalación de dependencias necesarias
# !pip install kaggle pandas sqlalchemy

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Rutas
REPO_ROOT = Path("..").resolve()
DATA_PATH = REPO_ROOT / "online_learning_engagement_dataset.csv"
DB_PATH   = REPO_ROOT / "online_learning.db"

## 2.2 Descarga del dataset mediante la API de Kaggle

La API de Kaggle permite descargar datasets con el comando `kaggle datasets download -d <owner>/<dataset>`.  
El dataset utilizado es: **`rabieelkharoua/students-performance-dataset`**  
*(Si el archivo ya existe localmente, se omite la descarga para no repetir el proceso.)*

In [ ]:
if not DATA_PATH.exists():
    import subprocess, zipfile

    # Dataset slug de Kaggle (owner/dataset-name)
    KAGGLE_DATASET = "rabieelkharoua/students-performance-dataset"

    print("Descargando dataset desde la API de Kaggle...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "-p", str(REPO_ROOT), "--unzip"],
        check=True
    )
    print(f"Dataset descargado en: {DATA_PATH}")
else:
    print(f"Dataset ya disponible en: {DATA_PATH}")

## 2.3 Carga del CSV con Pandas

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

In [ ]:
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Paso 3: Almacenamiento en Base de Datos

Se utiliza **SQLite** como motor de base de datos, gestionado a través de **SQLAlchemy** (ORM estudiado en el curso).  

### ¿Por qué SQLite?
- No requiere servidor: el archivo `.db` vive dentro del repositorio.
- Soporta SQL estándar completo: SELECT, JOIN, INSERT, UPDATE, DELETE...
- Totalmente compatible con SQLAlchemy y `pandas.read_sql`.
- Mayor seguridad y persistencia que un CSV plano.

El dataset se divide en **dos tablas relacionadas** para poder practicar JOINs:
- `students`: datos demográficos y de acceso.
- `performance`: métricas de rendimiento académico y engagement.

In [ ]:
from sqlalchemy import create_engine, text

# Crear engine SQLite (crea el archivo .db si no existe)
engine = create_engine(f"sqlite:///{DB_PATH}")

# --- Tabla 1: students (perfil demográfico y acceso) ---
students = df[["student_id", "age", "gender", "country",
               "device_type", "internet_speed_mbps",
               "study_hours_weekly", "login_frequency_weekly"]].copy()

# --- Tabla 2: performance (métricas académicas y de engagement) ---
performance = df[["student_id", "avg_session_duration_min",
                  "video_watch_time_min", "assignments_submitted",
                  "forum_posts", "quiz_attempts", "avg_quiz_score",
                  "attendance_rate", "engagement_score",
                  "final_grade", "dropout"]].copy()

# Guardar en SQLite (reemplaza si existe)
students.to_sql("students", engine, if_exists="replace", index=False)
performance.to_sql("performance", engine, if_exists="replace", index=False)

print("Tablas creadas correctamente en SQLite:")
with engine.connect() as conn:
    tablas = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    for t in tablas:
        print(f"  - {t[0]}")

## 3.1 Consultas SQL — SELECT

Consultas básicas para explorar los datos y obtener información de valor previo al EDA.

In [ ]:
# SELECT: Distribución de estudiantes por país (top 10)
query_paises = """
SELECT country,
       COUNT(*) AS total_students,
       ROUND(AVG(study_hours_weekly), 2) AS avg_study_hours
FROM students
GROUP BY country
ORDER BY total_students DESC
LIMIT 10
"""
df_paises = pd.read_sql(query_paises, engine)
print("Top 10 países por número de estudiantes:")
df_paises

In [ ]:
# SELECT: Tasa de abandono (dropout) por dispositivo
query_dropout = """
SELECT device_type,
       COUNT(*) AS total,
       SUM(p.dropout) AS dropouts,
       ROUND(100.0 * SUM(p.dropout) / COUNT(*), 2) AS dropout_rate_pct
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY device_type
ORDER BY dropout_rate_pct DESC
"""
df_dropout = pd.read_sql(query_dropout, engine)
print("Tasa de abandono por tipo de dispositivo:")
df_dropout

## 3.2 Consultas SQL — JOIN

Combinamos las tablas `students` y `performance` para obtener una visión completa del perfil del estudiante y su rendimiento.

In [ ]:
# JOIN: Perfil completo de estudiantes con alto riesgo de abandono
# (engagement_score bajo y attendance_rate < 0.5)
query_riesgo = """
SELECT s.student_id,
       s.age,
       s.gender,
       s.country,
       s.study_hours_weekly,
       p.engagement_score,
       p.attendance_rate,
       p.avg_quiz_score,
       p.final_grade,
       p.dropout
FROM students s
INNER JOIN performance p ON s.student_id = p.student_id
WHERE p.attendance_rate < 0.5
  AND p.engagement_score < 5
ORDER BY p.engagement_score ASC
LIMIT 15
"""
df_riesgo = pd.read_sql(query_riesgo, engine)
print(f"Estudiantes con alto riesgo de abandono: {len(df_riesgo)} registros (muestra de 15)")
df_riesgo

In [ ]:
# JOIN: Rendimiento promedio por género y velocidad de internet (alta vs baja)
query_internet = """
SELECT s.gender,
       CASE WHEN s.internet_speed_mbps >= 50 THEN 'Alta (>=50 Mbps)'
            ELSE 'Baja (<50 Mbps)' END AS internet_tier,
       COUNT(*) AS total_students,
       ROUND(AVG(p.final_grade), 2)      AS avg_final_grade,
       ROUND(AVG(p.avg_quiz_score), 2)   AS avg_quiz_score,
       ROUND(AVG(p.engagement_score), 2) AS avg_engagement
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY s.gender, internet_tier
ORDER BY s.gender, internet_tier
"""
df_internet = pd.read_sql(query_internet, engine)
print("Rendimiento según género y velocidad de conexión:")
df_internet

## 3.3 Consultas SQL — INSERT

Se inserta un nuevo registro de estudiante ficticio para demostrar la escritura en la base de datos.

In [ ]:
with engine.connect() as conn:
    # INSERT en tabla students
    conn.execute(text("""
        INSERT INTO students
            (student_id, age, gender, country, device_type,
             internet_speed_mbps, study_hours_weekly, login_frequency_weekly)
        VALUES
            (999999, 28, 'Male', 'Spain', 'Laptop', 100.0, 20.0, 7)
    """))

    # INSERT en tabla performance
    conn.execute(text("""
        INSERT INTO performance
            (student_id, avg_session_duration_min, video_watch_time_min,
             assignments_submitted, forum_posts, quiz_attempts,
             avg_quiz_score, attendance_rate, engagement_score,
             final_grade, dropout)
        VALUES
            (999999, 60.0, 500.0, 10, 5, 8, 85.0, 0.95, 9.5, 88.0, 0)
    """))

    conn.commit()
    print("INSERT completado. Verificando registro insertado:")

# Verificar el INSERT con SELECT
df_nuevo = pd.read_sql(
    "SELECT s.*, p.final_grade, p.dropout FROM students s JOIN performance p ON s.student_id = p.student_id WHERE s.student_id = 999999",
    engine
)
df_nuevo

## Base de datos completa

In [ ]:
df_full = pd.read_sql("""
    SELECT s.student_id, s.age, s.gender, s.country, s.device_type,
           s.internet_speed_mbps, s.study_hours_weekly, s.login_frequency_weekly,
           p.avg_session_duration_min, p.video_watch_time_min,
           p.assignments_submitted, p.forum_posts, p.quiz_attempts,
           p.avg_quiz_score, p.attendance_rate, p.engagement_score,
           p.final_grade, p.dropout
    FROM students s
    JOIN performance p ON s.student_id = p.student_id
    ORDER BY s.student_id
""", engine)

print(f"Total de registros en la base de datos: {len(df_full):,}")
print(f"Dimensiones: {df_full.shape[0]:,} filas × {df_full.shape[1]} columnas")
df_full

# Paso 4: Análisis Descriptivo

Antes de comenzar el EDA exploratorio, necesitamos conocer las **medidas estadísticas fundamentales** de cada variable predictora: medias, medianas, modas, desviaciones estándar, distribuciones, asimetría y curtosis.

Este análisis nos permitirá:
- Entender la tendencia central y dispersión de cada variable.
- Teorizar sobre la distribución que sigue cada una (normal, uniforme, sesgada, etc.).
- Aplicar **contrastes de hipótesis** (test de normalidad) para validar nuestras teorías.

Los datos se cargan directamente desde la base de datos SQLite (excluyendo el registro ficticio insertado en el paso anterior).

In [ ]:
## 4.1 Carga desde la base de datos y estadísticas descriptivas fundamentales

import numpy as np
from scipy import stats

# Cargar datos completos desde SQLite (sin el registro ficticio)
df_full = pd.read_sql("""
    SELECT s.student_id, s.age, s.gender, s.country, s.device_type,
           s.internet_speed_mbps, s.study_hours_weekly, s.login_frequency_weekly,
           p.avg_session_duration_min, p.video_watch_time_min,
           p.assignments_submitted, p.forum_posts, p.quiz_attempts,
           p.avg_quiz_score, p.attendance_rate, p.engagement_score,
           p.final_grade, p.dropout
    FROM students s
    JOIN performance p ON s.student_id = p.student_id
    WHERE s.student_id != 999999
    ORDER BY s.student_id
""", engine)

# Variables numéricas predictoras (excluyendo student_id y la target dropout)
num_cols = ["age", "internet_speed_mbps", "study_hours_weekly",
            "login_frequency_weekly", "avg_session_duration_min",
            "video_watch_time_min", "assignments_submitted", "forum_posts",
            "quiz_attempts", "avg_quiz_score", "attendance_rate",
            "engagement_score", "final_grade"]

# --- Estadísticas descriptivas completas ---
desc = df_full[num_cols].describe().T
desc["median"] = df_full[num_cols].median()
desc["mode"] = df_full[num_cols].mode().iloc[0]
desc["range"] = desc["max"] - desc["min"]
desc["IQR"] = desc["75%"] - desc["25%"]
desc["cv"] = (desc["std"] / desc["mean"] * 100).round(2)  # Coef. de variación (%)

print(f"Estadísticas descriptivas de {len(num_cols)} variables numéricas ({len(df_full):,} registros):\n")
desc[["count", "mean", "median", "mode", "std", "min", "25%", "50%", "75%", "max", "range", "IQR", "cv"]]

In [ ]:
## 4.2 Análisis de forma: asimetría (skewness) y curtosis

# Calcular skewness y kurtosis para cada variable
shape_analysis = pd.DataFrame({
    "skewness": df_full[num_cols].skew(),
    "kurtosis": df_full[num_cols].kurtosis()  # kurtosis excess (Fisher): normal = 0
})

# Interpretar la asimetría
def interpret_skew(s):
    if abs(s) < 0.5:
        return "Aprox. simétrica"
    elif s > 0:
        return "Sesgada a la derecha (+)"
    else:
        return "Sesgada a la izquierda (-)"

# Interpretar la curtosis (Fisher: normal = 0)
def interpret_kurt(k):
    if abs(k) < 0.5:
        return "Mesocúrtica (≈normal)"
    elif k > 0:
        return "Leptocúrtica (colas pesadas)"
    else:
        return "Platicúrtica (colas ligeras)"

shape_analysis["interp_skewness"] = shape_analysis["skewness"].apply(interpret_skew)
shape_analysis["interp_kurtosis"] = shape_analysis["kurtosis"].apply(interpret_kurt)

print("Análisis de forma de las distribuciones:\n")
shape_analysis

## 4.3 Visualización de distribuciones y contraste de normalidad

Para teorizar sobre la distribución de cada variable, visualizamos sus **histogramas con curva KDE** y aplicamos el **test de Kolmogorov-Smirnov** (más robusto que Shapiro-Wilk para muestras grandes, n > 5000).

**Hipótesis del test K-S:**
- **H₀**: La variable sigue una distribución normal.
- **H₁**: La variable NO sigue una distribución normal.
- **Criterio**: Si p-valor < 0.05, rechazamos H₀ (no es normal).

In [ ]:
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

ks_results = []

for i, col in enumerate(num_cols):
    ax = axes[i]
    data = df_full[col].dropna()

    # Histograma + KDE
    ax.hist(data, bins=40, density=True, alpha=0.6, color="steelblue", edgecolor="white")
    data.plot.kde(ax=ax, color="red", linewidth=2)

    # Test de Kolmogorov-Smirnov contra distribución normal
    data_std = (data - data.mean()) / data.std()
    ks_stat, ks_p = stats.kstest(data_std, "norm")
    normal = "Sí" if ks_p >= 0.05 else "No"

    ks_results.append({
        "variable": col,
        "KS_statistic": round(ks_stat, 4),
        "p_valor": f"{ks_p:.2e}",
        "normal_α0.05": normal
    })

    ax.set_title(f"{col}\nKS p={ks_p:.2e}", fontsize=10)
    ax.set_xlabel("")

# Ocultar ejes sobrantes
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de cada variable predictora (histograma + KDE)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Tabla resumen de los tests de normalidad
df_ks = pd.DataFrame(ks_results)
print("\nResultados del test de Kolmogorov-Smirnov (H₀: la variable es normal):\n")
df_ks

In [ ]:
## 4.4 Análisis de variables categóricas

cat_cols = ["gender", "country", "device_type"]

for col in cat_cols:
    print(f"\n{'='*50}")
    print(f"Variable: {col}")
    print(f"  Valores únicos: {df_full[col].nunique()}")
    print(f"  Moda: {df_full[col].mode()[0]}")
    print(f"\n  Frecuencias:")
    freq = df_full[col].value_counts()
    freq_pct = df_full[col].value_counts(normalize=True) * 100
    display(pd.DataFrame({"frecuencia": freq, "porcentaje (%)": freq_pct.round(2)}))

## 4.5 Teorización sobre las distribuciones

A partir del análisis descriptivo, la asimetría/curtosis, los histogramas y los tests de normalidad, podemos teorizar sobre la distribución de cada variable:

| Variable | Distribución teórica | Justificación |
|---|---|---|
| `age` | **Uniforme** | Rango acotado (18-40+), sin picos claros, baja curtosis |
| `internet_speed_mbps` | **Uniforme** | Distribuida homogéneamente en un rango continuo |
| `study_hours_weekly` | **Uniforme / Aprox. simétrica** | Dispersión regular sin sesgo fuerte |
| `login_frequency_weekly` | **Uniforme discreta** | Valores enteros en rango pequeño (1-7), distribución plana |
| `avg_session_duration_min` | **Uniforme** | Sin concentración en valores específicos |
| `video_watch_time_min` | **Uniforme** | Rango amplio sin picos dominantes |
| `assignments_submitted` | **Uniforme discreta** | Valores enteros distribuidos sin sesgo |
| `forum_posts` | **Uniforme discreta** | Sin concentración en extremos |
| `quiz_attempts` | **Uniforme discreta** | Distribución plana en rango acotado |
| `avg_quiz_score` | **Uniforme / Aprox. normal** | Dependerá del p-valor: si skewness ≈ 0 podría ser normal |
| `attendance_rate` | **Uniforme** | Proporción distribuida en [0,1] sin sesgo |
| `engagement_score` | **Uniforme** | Score sintético con dispersión homogénea |
| `final_grade` | **Uniforme / Aprox. normal** | Similar a quiz_score, evaluar con test K-S |

> **Nota**: Si el test K-S rechaza normalidad en la mayoría de variables (p < 0.05), esto es consistente con distribuciones uniformes generadas sintéticamente. Las variables de este dataset tienden a seguir distribuciones **uniformes continuas o discretas**, lo cual es típico de datasets simulados/sintéticos donde los valores se generan aleatoriamente dentro de rangos predefinidos.

# Paso 5: EDA Completo (Exploratory Data Analysis)

El objetivo de este EDA es asegurar que nos quedamos con las variables estrictamente necesarias y eliminamos las que no son relevantes o no aportan información.

**Problema a resolver**: Predecir si un estudiante abandonará el curso (`dropout` = 1) o lo completará (`dropout` = 0), a partir de sus datos demográficos, hábitos de estudio y métricas de rendimiento. Se trata de un problema de **clasificación binaria**.

**Sub-pasos del EDA:**
1. Planteamiento del problema y recopilación de datos
2. Exploración y limpieza de datos
3. Análisis de variables univariante
4. Análisis de variables multivariante
5. Ingeniería de características
6. Selección de características

## 5.1 Planteamiento del problema y recopilación de datos

**Variable objetivo (target):** `dropout` (0 = no abandona, 1 = abandona)

**Variables predictoras disponibles:**
- **Demográficas:** age, gender, country
- **Acceso tecnológico:** device_type, internet_speed_mbps
- **Hábitos de estudio:** study_hours_weekly, login_frequency_weekly
- **Engagement:** avg_session_duration_min, video_watch_time_min, forum_posts
- **Rendimiento:** assignments_submitted, quiz_attempts, avg_quiz_score, attendance_rate, engagement_score, final_grade

**Métricas de evaluación:** Al ser clasificación binaria con posible desbalanceo, usaremos Accuracy, Precision, Recall y F1-Score.

In [ ]:
# Recopilación: partimos del df_full ya cargado desde SQLite
# Eliminamos student_id (identificador, no predictora) y el registro ficticio
df_eda = df_full[df_full["student_id"] != 999999].drop(columns=["student_id"]).copy()

print(f"Dataset para EDA: {df_eda.shape[0]:,} filas × {df_eda.shape[1]} columnas")
print(f"\nVariable target: dropout")
print(f"  Distribución:\n{df_eda['dropout'].value_counts()}")
print(f"\n  Proporción de dropout: {df_eda['dropout'].mean()*100:.2f}%")
df_eda.head()

## 5.2 Exploración y limpieza de datos

Verificamos la calidad del dataset: valores nulos, duplicados, tipos de datos y outliers.

In [ ]:
# --- Valores nulos ---
print("Valores nulos por columna:")
print(df_eda.isnull().sum())
print(f"\nTotal de nulos en el dataset: {df_eda.isnull().sum().sum()}")

# --- Duplicados ---
duplicados = df_eda.duplicated().sum()
print(f"\nFilas duplicadas: {duplicados}")
if duplicados > 0:
    df_eda = df_eda.drop_duplicates()
    print(f"  → Eliminadas. Nuevo shape: {df_eda.shape}")

# --- Tipos de datos ---
print(f"\nTipos de datos:")
print(df_eda.dtypes)

In [ ]:
# --- Detección de outliers con IQR ---
num_cols_eda = df_eda.select_dtypes(include=[np.number]).columns.tolist()
num_cols_eda = [c for c in num_cols_eda if c != "dropout"]  # excluir target

outlier_summary = []
for col in num_cols_eda:
    Q1 = df_eda[col].quantile(0.25)
    Q3 = df_eda[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_eda[col] < lower) | (df_eda[col] > upper)).sum()
    outlier_summary.append({
        "variable": col,
        "Q1": round(Q1, 2),
        "Q3": round(Q3, 2),
        "IQR": round(IQR, 2),
        "lim_inferior": round(lower, 2),
        "lim_superior": round(upper, 2),
        "n_outliers": n_outliers,
        "pct_outliers": round(n_outliers / len(df_eda) * 100, 2)
    })

df_outliers = pd.DataFrame(outlier_summary)
print("Detección de outliers (método IQR × 1.5):\n")
df_outliers

In [ ]:
# Boxplots para visualizar outliers
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    df_eda.boxplot(column=col, ax=axes[i], vert=True)
    axes[i].set_title(col, fontsize=10)

for j in range(len(num_cols_eda), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots — Detección visual de outliers", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5.3 Análisis de variables univariante

Analizamos cada variable de forma individual para entender su distribución y su relación con la variable target (`dropout`).

In [ ]:
import seaborn as sns

# --- Variables numéricas: histogramas separados por dropout ---
fig, axes = plt.subplots(4, 4, figsize=(22, 18))
axes = axes.flatten()

for i, col in enumerate(num_cols_eda):
    ax = axes[i]
    for label, color in [(0, "steelblue"), (1, "salmon")]:
        subset = df_eda[df_eda["dropout"] == label][col]
        ax.hist(subset, bins=35, alpha=0.5, color=color, label=f"dropout={label}", density=True)
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

for j in range(len(num_cols_eda), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de variables numéricas segmentada por dropout", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Variables categóricas: countplots segmentados por dropout ---
cat_cols = ["gender", "device_type"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(cat_cols):
    sns.countplot(data=df_eda, x=col, hue="dropout", ax=axes[i], palette=["steelblue", "salmon"])
    axes[i].set_title(f"Distribución de {col} por dropout", fontsize=12)
    axes[i].legend(title="dropout")

plt.tight_layout()
plt.show()

# Country tiene muchos valores únicos → mostramos tasa de dropout por país (top 10 países)
top_countries = df_eda["country"].value_counts().head(10).index
df_country_dropout = df_eda[df_eda["country"].isin(top_countries)].groupby("country")["dropout"].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 5))
df_country_dropout.plot(kind="bar", color="salmon", edgecolor="white", ax=ax)
ax.set_title("Tasa de dropout (%) por país (top 10 países por volumen)", fontsize=12)
ax.set_ylabel("Dropout (%)")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5.4 Análisis de variables multivariante

Analizamos las relaciones entre variables para detectar multicolinealidad y entender qué variables están más asociadas con el dropout.

In [ ]:
# --- Matriz de correlación ---
corr_cols = num_cols_eda + ["dropout"]
corr_matrix = df_eda[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Matriz de correlación (Pearson)", fontsize=14)
plt.tight_layout()
plt.show()

# Correlación con la variable target
print("\nCorrelación de cada variable con dropout (ordenada por valor absoluto):\n")
corr_target = corr_matrix["dropout"].drop("dropout").abs().sort_values(ascending=False)
corr_target_df = pd.DataFrame({
    "correlación": corr_matrix["dropout"].drop("dropout"),
    "|correlación|": corr_target
}).sort_values("|correlación|", ascending=False)
corr_target_df

In [ ]:
# --- Detección de multicolinealidad (pares con |r| > 0.8) ---
import itertools

print("Pares de variables con alta multicolinealidad (|r| > 0.8):\n")
high_corr_pairs = []
for col1, col2 in itertools.combinations(num_cols_eda, 2):
    r = corr_matrix.loc[col1, col2]
    if abs(r) > 0.8:
        high_corr_pairs.append({"var_1": col1, "var_2": col2, "correlación": round(r, 4)})
        print(f"  {col1} ↔ {col2}: r = {r:.4f}")

if not high_corr_pairs:
    print("  No se encontraron pares con |r| > 0.8")

# --- Pairplot de las variables más correlacionadas con dropout (top 5) ---
top5_vars = corr_target.head(5).index.tolist() + ["dropout"]
sns.pairplot(df_eda[top5_vars], hue="dropout", palette=["steelblue", "salmon"],
             diag_kind="kde", plot_kws={"alpha": 0.3, "s": 10})
plt.suptitle("Pairplot — Top 5 variables más correlacionadas con dropout", y=1.02, fontsize=14)
plt.show()

## 5.5 Ingeniería de características

Transformamos las variables categóricas en numéricas y creamos nuevas features que puedan aportar información al modelo.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_fe = df_eda.copy()

# --- Encoding de variables categóricas ---
# Gender: Label Encoding (binario)
le_gender = LabelEncoder()
df_fe["gender_encoded"] = le_gender.fit_transform(df_fe["gender"])
print(f"Gender encoding: {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")

# Device type: One-Hot Encoding (pocas categorías, no ordinal)
df_fe = pd.get_dummies(df_fe, columns=["device_type"], prefix="device", drop_first=True)
print(f"Device type → One-Hot Encoding (drop_first=True)")

# Country: muchos valores únicos → frecuencia relativa como proxy
country_freq = df_fe["country"].value_counts(normalize=True)
df_fe["country_freq"] = df_fe["country"].map(country_freq)
print(f"Country → Frequency Encoding ({df_fe['country'].nunique()} países)")

# --- Nuevas features ---
# Ratio de eficiencia: nota final / horas de estudio
df_fe["grade_per_hour"] = df_fe["final_grade"] / (df_fe["study_hours_weekly"] + 1)

# Ratio de participación activa: forum_posts / login_frequency
df_fe["participation_ratio"] = df_fe["forum_posts"] / (df_fe["login_frequency_weekly"] + 1)

# Tiempo promedio productivo: (video + session) / login_frequency
df_fe["productive_time"] = (df_fe["video_watch_time_min"] + df_fe["avg_session_duration_min"]) / (df_fe["login_frequency_weekly"] + 1)

# Eliminar columnas originales categóricas (ya codificadas)
df_fe = df_fe.drop(columns=["gender", "country"])

print(f"\nDataset tras ingeniería de características: {df_fe.shape[0]:,} filas × {df_fe.shape[1]} columnas")
print(f"Nuevas columnas: gender_encoded, country_freq, grade_per_hour, participation_ratio, productive_time")
df_fe.head()

## 5.6 Selección de características

Seleccionamos las variables más relevantes para el modelo eliminando las que no aportan información o son redundantes. Usamos la importancia de un modelo de árbol (Random Forest) como criterio objetivo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Separar features y target
X = df_fe.drop(columns=["dropout"])
y = df_fe["dropout"]

# Feature importance con Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

feat_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

# Visualización
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=feat_importance, x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title("Importancia de características (Random Forest)", fontsize=14)
ax.set_xlabel("Importancia")
plt.tight_layout()
plt.show()

print("\nRanking de importancia:\n")
feat_importance

In [ ]:
# --- Selección: eliminar features con importancia < 1% ---
threshold = 0.01
selected_features = feat_importance[feat_importance["importance"] >= threshold]["feature"].tolist()
dropped_features = feat_importance[feat_importance["importance"] < threshold]["feature"].tolist()

print(f"Umbral de importancia: {threshold}")
print(f"Features seleccionadas ({len(selected_features)}): {selected_features}")
print(f"Features eliminadas ({len(dropped_features)}): {dropped_features}")

X_selected = X[selected_features]
print(f"\nDataset final para modelado: {X_selected.shape[0]:,} filas × {X_selected.shape[1]} features + target")

In [ ]:
# --- División train/test ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

print(f"División train/test (80/20, estratificada por dropout):")
print(f"  Train: {X_train.shape[0]:,} filas × {X_train.shape[1]} features")
print(f"  Test:  {X_test.shape[0]:,} filas × {X_test.shape[1]} features")
print(f"\n  Proporción dropout en train: {y_train.mean()*100:.2f}%")
print(f"  Proporción dropout en test:  {y_test.mean()*100:.2f}%")
print(f"\n  → La estratificación mantiene la proporción de la clase target en ambos conjuntos.")

# Paso 6: Construcción y Optimización del Modelo

Se trata de un problema de **clasificación binaria** (predecir `dropout`). Se prueban tres modelos con perfiles complementarios:

| Modelo | Por qué lo incluimos |
|---|---|
| **Logistic Regression** | Modelo de referencia (baseline). Lineal, muy interpretable. Nos permite saber si el problema tiene separabilidad lineal. Rápido de entrenar, sirve como cota mínima de rendimiento. |
| **Random Forest** | Ensemble de árboles con bagging. Robusto ante outliers y variables de escala diferente. Captura relaciones no lineales sin necesitar normalización. Ya demostró utilidad en la selección de features. |
| **Gradient Boosting** | Ensemble secuencial que corrige errores iterativamente. Suele superar a Random Forest en datos tabulares estructurados. Más sensible a hiperparámetros pero con mayor potencial de rendimiento.|

**Estrategia de optimización:**
- Logistic Regression → `GridSearchCV` (espacio pequeño)
- Random Forest y Gradient Boosting → `RandomizedSearchCV` (espacio grande, más eficiente)
- Métrica principal: **F1-Score** (equilibrio entre Precision y Recall, importante cuando el coste de falsos negativos es alto — un dropout no detectado)

## 6.1 Evaluación baseline (sin optimizar)

Entrenamos los tres modelos con sus parámetros por defecto para establecer una línea base de comparación.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

# Escalar para Logistic Regression (sensible a la escala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest":       RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

baseline_results = []

for name, model in baseline_models.items():
    X_tr = X_train_scaled if name == "Logistic Regression" else X_train
    X_te = X_test_scaled  if name == "Logistic Regression" else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    baseline_results.append({
        "Modelo": name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob), 4),
    })

df_baseline = pd.DataFrame(baseline_results).set_index("Modelo")
print("Resultados baseline (sin optimizar):\n")
df_baseline

## 6.2 Optimización — Logistic Regression (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score

# Grid de hiperparámetros para Logistic Regression
param_grid_lr = {
    "C":       [0.01, 0.1, 1, 10, 100],       # Inverso de la regularización
    "penalty": ["l1", "l2"],                   # Tipo de regularización
    "solver":  ["liblinear", "saga"],          # Solvers compatibles con l1
}

gs_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid_lr,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=0
)
gs_lr.fit(X_train_scaled, y_train)

print(f"Mejores hiperparámetros: {gs_lr.best_params_}")
print(f"Mejor F1-Score (CV): {gs_lr.best_score_:.4f}")

# Evaluación en test
best_lr = gs_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test_scaled)
y_prob_lr = best_lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nResultados en test (Logistic Regression optimizada):")
print(classification_report(y_test, y_pred_lr, target_names=["No dropout", "Dropout"]))

## 6.3 Optimización — Random Forest (RandomizedSearchCV)

In [ ]:
from scipy.stats import randint as sp_randint

param_dist_rf = {
    "n_estimators":      sp_randint(100, 500),
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": sp_randint(2, 20),
    "min_samples_leaf":  sp_randint(1, 10),
    "max_features":      ["sqrt", "log2", None],
    "class_weight":      [None, "balanced"],
}

rs_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_rf,
    n_iter=50,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rs_rf.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {rs_rf.best_params_}")
print(f"Mejor F1-Score (CV): {rs_rf.best_score_:.4f}")

best_rf = rs_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]

print(f"\nResultados en test (Random Forest optimizado):")
print(classification_report(y_test, y_pred_rf, target_names=["No dropout", "Dropout"]))

## 6.4 Optimización — Gradient Boosting (RandomizedSearchCV)

In [ ]:
from scipy.stats import uniform as sp_uniform

param_dist_gb = {
    "n_estimators":   sp_randint(100, 400),
    "learning_rate":  sp_uniform(0.01, 0.3),
    "max_depth":      sp_randint(3, 8),
    "subsample":      sp_uniform(0.6, 0.4),      # 0.6 – 1.0
    "min_samples_split": sp_randint(2, 20),
    "max_features":   ["sqrt", "log2", None],
}

rs_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist_gb,
    n_iter=50,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rs_gb.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {rs_gb.best_params_}")
print(f"Mejor F1-Score (CV): {rs_gb.best_score_:.4f}")

best_gb = rs_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test)
y_prob_gb = best_gb.predict_proba(X_test)[:, 1]

print(f"\nResultados en test (Gradient Boosting optimizado):")
print(classification_report(y_test, y_pred_gb, target_names=["No dropout", "Dropout"]))

## 6.5 Comparación final de los tres modelos optimizados

In [ ]:
results_opt = [
    ("Logistic Regression", y_pred_lr, y_prob_lr),
    ("Random Forest",       y_pred_rf, y_prob_rf),
    ("Gradient Boosting",   y_pred_gb, y_prob_gb),
]

comparison = []
for name, y_pred, y_prob in results_opt:
    comparison.append({
        "Modelo":    name,
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1-Score":  round(f1_score(y_test, y_pred), 4),
        "ROC-AUC":   round(roc_auc_score(y_test, y_prob), 4),
    })

df_comparison = pd.DataFrame(comparison).set_index("Modelo")
print("Comparación final — Modelos optimizados:\n")
display(df_comparison)

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(10, 5))
df_comparison[["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]].T.plot(
    kind="bar", ax=ax, colormap="viridis", edgecolor="white"
)
ax.set_title("Comparación de métricas — Modelos optimizados", fontsize=13)
ax.set_ylabel("Valor")
ax.set_ylim(0, 1.05)
ax.legend(title="Modelo")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred, _) in zip(axes, results_opt):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No dropout", "Dropout"],
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(name, fontsize=12)
plt.suptitle("Matrices de confusión — Modelos optimizados", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
## 6.6 Conclusión — Selección del modelo ganador

# Identificar automáticamente el ganador por F1-Score
winner = df_comparison["F1-Score"].idxmax()
winner_f1  = df_comparison.loc[winner, "F1-Score"]
winner_auc = df_comparison.loc[winner, "ROC-AUC"]
winner_acc = df_comparison.loc[winner, "Accuracy"]

print(f"Modelo ganador: {winner}")
print(f"  F1-Score : {winner_f1}")
print(f"  ROC-AUC  : {winner_auc}")
print(f"  Accuracy : {winner_acc}")

# Curvas ROC de los tres modelos
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["steelblue", "darkorange", "seagreen"]
for (name, _, y_prob), color in zip(results_opt, colors):
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax, color=color)
ax.plot([0, 1], [0, 1], "k--", label="Random classifier")
ax.set_title("Curvas ROC — Modelos optimizados", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## Conclusión del Paso 6

### ¿Por qué estos tres modelos?

**Logistic Regression** se incluyó como modelo de referencia (baseline). Al ser lineal, nos permite verificar si existe una separación lineal entre dropout y no-dropout. Si un modelo complejo no supera significativamente a la regresión logística, el problema podría resolverse con un enfoque más simple e interpretable.

**Random Forest** se eligió porque es un ensemble robusto que maneja bien datasets tabulares de tamaño medio, no requiere normalización y ya había demostrado utilidad durante la selección de features. Su capacidad para capturar relaciones no lineales sin sobreajuste lo convierte en un candidato sólido para clasificación binaria.

**Gradient Boosting** se incluyó por ser el algoritmo que típicamente ofrece el mayor rendimiento en datos tabulares estructurados. A diferencia del Random Forest (bagging en paralelo), el Gradient Boosting construye árboles secuencialmente corrigiendo los errores del modelo anterior, lo que lo hace más preciso pero también más sensible a sus hiperparámetros — de ahí la importancia del paso de optimización.

### ¿Por qué el modelo ganador?

El **modelo ganador es el que obtuvo el mayor F1-Score** en el conjunto de test tras la optimización de hiperparámetros. Se priorizó el F1-Score (en lugar de Accuracy) porque en el contexto del abandono escolar, **un falso negativo tiene un coste alto**: predecir que un estudiante no va a abandonar cuando sí lo hará implica no intervenir a tiempo. El F1-Score equilibra precisión y recall, penalizando ambos tipos de error de forma proporcional.

La curva ROC y el ROC-AUC complementan este análisis mostrando la capacidad discriminativa del modelo a distintos umbrales de clasificación.